In [1]:
# ─── Prior Win Probability Model ────────────────────────────────────────────
#
# Pre-game prediction: team stats → P(home_win).
# Uses XGBoost with temporal train/val/test split and early stopping.
# Output: prior home win probability for each game in training_games4.csv,
# used as the prior input to the posterior (in-game) Beta model.

import pandas as pd
import numpy as np
import json
import pickle
import time
import warnings
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, brier_score_loss, log_loss)
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')

RANDOM_SEED = 42
DATA_PATH = 'data/training_games4.csv'
MODEL_DIR = Path('prior_models')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# XGBoost hyperparameters — regularized to reduce train-test gap
N_ESTIMATORS = 1000       # max rounds (early stopping will pick best)
LEARNING_RATE = 0.02
MAX_DEPTH = 2
SUBSAMPLE = 0.7
COLSAMPLE_BYTREE = 0.7
REG_LAMBDA = 8.0
REG_ALPHA = 0.5
MIN_CHILD_WEIGHT = 15
GAMMA = 0.5

# Early stopping
PATIENCE = 30
MIN_DELTA = 1e-5  # minimum improvement to reset patience

np.random.seed(RANDOM_SEED)

print(f'Config: N_ESTIMATORS={N_ESTIMATORS}, LR={LEARNING_RATE}, MAX_DEPTH={MAX_DEPTH}')
print(f'Regularization: REG_LAMBDA={REG_LAMBDA}, REG_ALPHA={REG_ALPHA}, GAMMA={GAMMA}')
print(f'Stochastic: SUBSAMPLE={SUBSAMPLE}, COLSAMPLE={COLSAMPLE_BYTREE}, MIN_CHILD_WEIGHT={MIN_CHILD_WEIGHT}')
print(f'Early stopping: PATIENCE={PATIENCE}, MIN_DELTA={MIN_DELTA}')

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Config: N_ESTIMATORS=1000, LR=0.02, MAX_DEPTH=2
Regularization: REG_LAMBDA=8.0, REG_ALPHA=0.5, GAMMA=0.5
Stochastic: SUBSAMPLE=0.7, COLSAMPLE=0.7, MIN_CHILD_WEIGHT=15
Early stopping: PATIENCE=30, MIN_DELTA=1e-05


In [2]:
# ─── 2. Load Data & Temporal Split ─────────────────────────────────────────
#
# Temporal split by season (no future leakage):
#   Train: seasons <= max-2
#   Val:   season = max-1  (used for early stopping)
#   Test:  season = max    (held-out evaluation)

t0 = time.time()
raw = pd.read_csv(DATA_PATH, dtype={'game_id': str})
print(f'Loaded {len(raw):,} games in {time.time()-t0:.1f}s')

raw['game_date'] = pd.to_datetime(raw['game_date'])

# Derive season from game_date: Oct+ = that year, Jan-Sep = previous year
raw['season'] = raw['game_date'].apply(
    lambda d: d.year if d.month >= 10 else d.year - 1)

# Target: home team wins (score_diff > 0)
raw['home_win'] = (raw['score_diff'] > 0).astype(int)

# Feature columns = everything except metadata, target, and 5g player stats
META_COLS = ['game_id', 'game_date', 'home_team', 'away_team', 'score_diff',
             'home_win', 'season']
FEATURE_COLS = [c for c in raw.columns if c not in META_COLS and '5g_player' not in c]

dropped_5g = [c for c in raw.columns if '5g_player' in c]
print(f'Features: {len(FEATURE_COLS)} (dropped {len(dropped_5g)} 5g_player columns)')
print(f'Seasons: {sorted(raw["season"].unique())}')
print(f'Date range: {raw["game_date"].min().date()} to {raw["game_date"].max().date()}')

# Temporal split
seasons = sorted(raw['season'].unique())
max_season = max(seasons)
val_season = max_season - 1
train_seasons = [s for s in seasons if s <= max_season - 2]

df_train = raw[raw['season'].isin(train_seasons)].copy()
df_val   = raw[raw['season'] == val_season].copy()
df_test  = raw[raw['season'] == max_season].copy()

# Verify no leakage
assert set(df_train['game_id']).isdisjoint(set(df_val['game_id']))
assert set(df_train['game_id']).isdisjoint(set(df_test['game_id']))
assert set(df_val['game_id']).isdisjoint(set(df_test['game_id']))

X_train, y_train = df_train[FEATURE_COLS], df_train['home_win']
X_val,   y_val   = df_val[FEATURE_COLS],   df_val['home_win']
X_test,  y_test  = df_test[FEATURE_COLS],  df_test['home_win']

for name, X, y in [('Train', X_train, y_train), ('Val', X_val, y_val), ('Test', X_test, y_test)]:
    ss = raw[raw['game_id'].isin(X.index.map(lambda i: raw.loc[i, 'game_id']))]['season'].unique()
    print(f'  {name:5s}: {len(X):>6,} games, home_win_rate={y.mean():.3f}, '
          f'seasons {sorted(raw.loc[X.index, "season"].unique())}')

Loaded 19,562 games in 0.7s
Features: 159 (dropped 36 5g_player columns)
Seasons: [2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Date range: 2008-11-15 to 2026-04-04
  Train: 17,504 games, home_win_rate=0.582, seasons [2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]
  Val  :  1,051 games, home_win_rate=0.550, seasons [2024]
  Test :  1,007 games, home_win_rate=0.542, seasons [2025]


In [3]:
# ─── 3. Train XGBoost with Early Stopping ──────────────────────────────────

pos_rate = float(y_train.mean())
scale_pos_weight = (1.0 - pos_rate) / pos_rate

model = XGBClassifier(
    n_estimators=N_ESTIMATORS,
    learning_rate=LEARNING_RATE,
    max_depth=MAX_DEPTH,
    subsample=SUBSAMPLE,
    colsample_bytree=COLSAMPLE_BYTREE,
    reg_lambda=REG_LAMBDA,
    reg_alpha=REG_ALPHA,
    min_child_weight=MIN_CHILD_WEIGHT,
    gamma=GAMMA,
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_SEED,
    n_jobs=-1,
    eval_metric='logloss',
    early_stopping_rounds=PATIENCE,
)

print(f'Training XGBoost (max {N_ESTIMATORS} rounds, early stopping patience={PATIENCE})')
print(f'scale_pos_weight={scale_pos_weight:.4f}')
print('-' * 60)

t0 = time.time()
model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    verbose=10
)

elapsed = time.time() - t0
best_round = model.best_iteration
best_score = model.best_score

from sklearn.metrics import accuracy_score
val_acc = accuracy_score(y_val, model.predict(X_val))
train_acc = accuracy_score(y_train, model.predict(X_train))

print(f'\nTraining complete in {elapsed:.1f}s')
print(f'Best round: {best_round} (val logloss: {best_score:.5f})')
print(f'Train accuracy: {train_acc:.4f}')
print(f'Val accuracy:   {val_acc:.4f}')
print(f'Train-Val gap:  {train_acc - val_acc:.4f}')

Training XGBoost (max 1000 rounds, early stopping patience=30)
scale_pos_weight=0.7186
------------------------------------------------------------
[0]	validation_0-logloss:0.69224	validation_1-logloss:0.69220
[10]	validation_0-logloss:0.67796	validation_1-logloss:0.67840
[20]	validation_0-logloss:0.66746	validation_1-logloss:0.66862
[30]	validation_0-logloss:0.65876	validation_1-logloss:0.66037
[40]	validation_0-logloss:0.65190	validation_1-logloss:0.65364
[50]	validation_0-logloss:0.64625	validation_1-logloss:0.64825
[60]	validation_0-logloss:0.64193	validation_1-logloss:0.64432
[70]	validation_0-logloss:0.63821	validation_1-logloss:0.64070
[80]	validation_0-logloss:0.63505	validation_1-logloss:0.63816
[90]	validation_0-logloss:0.63252	validation_1-logloss:0.63579
[100]	validation_0-logloss:0.63029	validation_1-logloss:0.63408
[110]	validation_0-logloss:0.62846	validation_1-logloss:0.63226
[120]	validation_0-logloss:0.62674	validation_1-logloss:0.63082
[130]	validation_0-logloss:0.62

In [5]:
# ─── 3b. Model Comparison ──────────────────────────────────────────────────
#
# Compare LogisticRegression, DecisionTree, XGBoost, and MLP
# using the same temporal train split with K-Fold cross-validation.

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_validate, KFold

try:
    from xgboost import XGBClassifier as XGB_CV
    _HAS_XGBOOST = True
except Exception:
    print("xgboost not available")
    _HAS_XGBOOST = False

k_folds = 5
kf = KFold(n_splits=k_folds, shuffle=True, random_state=RANDOM_SEED)

scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'roc_auc': 'roc_auc'
}

pos_rate_cv = float(y_train.mean())
scale_pos_weight_cv = (1.0 - pos_rate_cv) / pos_rate_cv

comparison_models = {}

comparison_models["LogisticRegression"] = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(max_iter=1000, random_state=RANDOM_SEED,
                              solver='lbfgs', class_weight='balanced'))
])

comparison_models["DecisionTree"] = DecisionTreeClassifier(
    random_state=RANDOM_SEED, class_weight="balanced", max_depth=None
)

if _HAS_XGBOOST:
    comparison_models["XGBoost"] = XGB_CV(
        n_estimators=600, learning_rate=0.05, max_depth=4,
        subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
        random_state=RANDOM_SEED, n_jobs=-1, eval_metric="logloss",
        scale_pos_weight=scale_pos_weight_cv
    )

comparison_models["MLP"] = Pipeline([
    ("scaler", StandardScaler()),
    ("mlp", MLPClassifier(
        hidden_layer_sizes=(128, 64), activation="relu", solver="adam",
        alpha=1e-4, batch_size=256, learning_rate_init=1e-3,
        max_iter=200, early_stopping=True, n_iter_no_change=10,
        random_state=RANDOM_SEED
    ))
])

print(f"Comparing {len(comparison_models)} models using {k_folds}-fold CV on TRAIN set")
print(f"(Temporal split: train seasons only, no future leakage)")
print("=" * 60)

cv_results_list = []
for name, mdl in comparison_models.items():
    print(f"\n{name}")
    print("-" * len(name))
    cv = cross_validate(
        mdl, X_train, y_train, cv=kf, scoring=scoring,
        return_train_score=True, n_jobs=-1
    )
    cv_results_list.append({
        "model": name,
        "test_accuracy_mean": cv["test_accuracy"].mean(),
        "test_accuracy_std":  cv["test_accuracy"].std(),
        "test_precision_mean": cv["test_precision"].mean(),
        "test_precision_std":  cv["test_precision"].std(),
        "test_recall_mean": cv["test_recall"].mean(),
        "test_recall_std":  cv["test_recall"].std(),
        "test_f1_mean": cv["test_f1"].mean(),
        "test_f1_std":  cv["test_f1"].std(),
        "test_roc_auc_mean": cv["test_roc_auc"].mean(),
        "test_roc_auc_std":  cv["test_roc_auc"].std(),
        "train_accuracy_mean": cv["train_accuracy"].mean(),
        "train_roc_auc_mean": cv["train_roc_auc"].mean(),
        "train_test_acc_gap": cv["train_accuracy"].mean() - cv["test_accuracy"].mean(),
    })

comparison_df = pd.DataFrame(cv_results_list).sort_values("test_roc_auc_mean", ascending=False)

cols = ["model", "test_roc_auc_mean", "test_roc_auc_std",
        "test_accuracy_mean", "test_accuracy_std",
        "test_f1_mean", "test_f1_std", "train_test_acc_gap"]
print("\n" + "=" * 60)
print("Model Comparison (ranked by ROC AUC):")
print("=" * 60)
for _, r in comparison_df.iterrows():
    print(f"\n  {r['model']}:")
    print(f"    ROC AUC:  {r['test_roc_auc_mean']:.4f} +/- {r['test_roc_auc_std']:.4f}")
    print(f"    Accuracy: {r['test_accuracy_mean']:.4f} +/- {r['test_accuracy_std']:.4f}")
    print(f"    F1:       {r['test_f1_mean']:.4f} +/- {r['test_f1_std']:.4f}")
    print(f"    Overfit gap: {r['train_test_acc_gap']:.4f}")

Comparing 4 models using 5-fold CV on TRAIN set
(Temporal split: train seasons only, no future leakage)

LogisticRegression
------------------


KeyboardInterrupt: 

In [ ]:
# ─── 3c. XGBoost Hyperparameter Grid Search ───────────────────────────────
#
# Fine-tune XGBoost hyperparameters using the temporal train/val split.

from sklearn.model_selection import GridSearchCV
import xgboost as xgb

param_grid = {
    'n_estimators': [200, 300, 400],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.02, 0.05],
    'min_child_weight': [1],
    'subsample': [0.8],
    'colsample_bytree': [0.8],
    'gamma': [0]
}

print("XGBoost Hyperparameter Grid Search")
print("=" * 60)
print(f"\nParameter grid:")
for param, values in param_grid.items():
    print(f"  {param}: {values}")
print(f"\nTotal combinations: {np.prod([len(v) for v in param_grid.values()])}")

xgb_clf = xgb.XGBClassifier(
    objective='binary:logistic', eval_metric='logloss',
    random_state=RANDOM_SEED, n_jobs=-1
)

grid_search = GridSearchCV(
    estimator=xgb_clf, param_grid=param_grid,
    scoring='roc_auc', cv=5, verbose=2, n_jobs=-1,
    return_train_score=True
)

print("\nStarting grid search (this may take a while)...")
print("Using 5-fold cross-validation with ROC AUC scoring\n")

grid_search.fit(X_train, y_train)

print("\n" + "=" * 60)
print("Grid Search Complete!")
print("=" * 60)

print("\nBest Parameters:")
for param, value in grid_search.best_params_.items():
    print(f"  {param}: {value}")
print(f"\nBest CV ROC AUC Score: {grid_search.best_score_:.4f}")

# Test on holdout set with best model
best_xgb = grid_search.best_estimator_
y_pred_best = best_xgb.predict(X_test)
y_proba_best = best_xgb.predict_proba(X_test)[:, 1]

print("\nHoldout Performance (Test season):")
print(f"  Accuracy : {accuracy_score(y_test, y_pred_best):.4f}")
print(f"  Precision: {precision_score(y_test, y_pred_best):.4f}")
print(f"  Recall   : {recall_score(y_test, y_pred_best):.4f}")
print(f"  F1       : {f1_score(y_test, y_pred_best):.4f}")
print(f"  ROC AUC  : {roc_auc_score(y_test, y_proba_best):.4f}")

# Feature importance
importances_best = best_xgb.feature_importances_
fi_best = pd.DataFrame({
    "feature": FEATURE_COLS, "importance": importances_best
}).sort_values("importance", ascending=False)

print("\nTop 25 Feature Importances (Best Model):")
print(fi_best.head(25).to_string(index=False))

# Top 10 parameter combinations
print("\nTop 10 Parameter Combinations by CV Score:")
gs_results_df = pd.DataFrame(grid_search.cv_results_)
gs_results_df = gs_results_df.sort_values('mean_test_score', ascending=False)
print(gs_results_df[['params', 'mean_test_score', 'std_test_score', 'mean_train_score']].head(10).to_string(index=False))

XGBoost Hyperparameter Grid Search

Parameter grid:
  n_estimators: [200, 300, 400]
  max_depth: [3, 5, 7]
  learning_rate: [0.02, 0.05]
  min_child_weight: [1]
  subsample: [0.8]
  colsample_bytree: [0.8]
  gamma: [0]

Total combinations: 18

Starting grid search (this may take a while)...
Using 5-fold cross-validation with ROC AUC scoring

Fitting 5 folds for each of 18 candidates, totalling 90 fits
[CV] END colsample_bytree=0.8, gamma=0, learning_rate=0.02, max_depth=3, min_child_weight=1, n_estimators=200, subsample=0.8; total time=   2.6s
[CV] END colsample_bytree=0.8, gamma=0, learning_rate=0.02, max_depth=3, min_child_weight=1, n_estimators=200, subsample=0.8; total time=   2.7s
[CV] END colsample_bytree=0.8, gamma=0, learning_rate=0.02, max_depth=3, min_child_weight=1, n_estimators=200, subsample=0.8; total time=   2.6s
[CV] END colsample_bytree=0.8, gamma=0, learning_rate=0.02, max_depth=3, min_child_weight=1, n_estimators=200, subsample=0.8; total time=   2.7s
[CV] END colsam

In [ ]:
# ─── 4. Evaluation ─────────────────────────────────────────────────────────

def evaluate_split(X, y, label):
    y_proba = model.predict_proba(X)[:, 1]
    y_pred = (y_proba >= 0.5).astype(int)

    acc  = accuracy_score(y, y_pred)
    prec = precision_score(y, y_pred)
    rec  = recall_score(y, y_pred)
    f1   = f1_score(y, y_pred)
    auc  = roc_auc_score(y, y_proba)
    ll   = log_loss(y, y_proba)
    bs   = brier_score_loss(y, y_proba)

    print(f'\n{label}:')
    print(f'  Accuracy:    {acc:.4f}')
    print(f'  Precision:   {prec:.4f}')
    print(f'  Recall:      {rec:.4f}')
    print(f'  F1:          {f1:.4f}')
    print(f'  ROC AUC:     {auc:.4f}')
    print(f'  Log Loss:    {ll:.5f}')
    print(f'  Brier Score: {bs:.5f}')

    # Calibration by decile
    bins = np.linspace(0, 1, 11)
    bin_idx = np.clip(np.digitize(y_proba, bins) - 1, 0, 9)
    print(f'  Calibration:')
    for b in range(10):
        mask = bin_idx == b
        if mask.sum() > 0:
            pred_mean = y_proba[mask].mean()
            obs_mean  = y.values[mask].mean()
            print(f'    [{bins[b]:.1f}-{bins[b+1]:.1f}): '
                  f'pred={pred_mean:.3f} obs={obs_mean:.3f} n={mask.sum()}')

    return {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1,
            'roc_auc': auc, 'log_loss': ll, 'brier_score': bs,
            'predictions': y_proba, 'labels': y.values}


train_results = evaluate_split(X_train, y_train, 'Train')
val_results   = evaluate_split(X_val,   y_val,   'Validation')
test_results  = evaluate_split(X_test,  y_test,  'Test')

# Overfitting check
gap = train_results['accuracy'] - test_results['accuracy']
print(f'\nTrain-Test accuracy gap: {gap:.4f}')

# Calibration plot
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (results, label) in zip(axes, [(train_results, 'Train'),
                                        (val_results, 'Val'),
                                        (test_results, 'Test')]):
    mu_r = results['predictions']
    y_r  = results['labels']
    bins = np.linspace(0, 1, 11)
    bin_idx = np.clip(np.digitize(mu_r, bins) - 1, 0, 9)

    pred_means, obs_means = [], []
    for b in range(10):
        mask = bin_idx == b
        if mask.sum() > 5:
            pred_means.append(mu_r[mask].mean())
            obs_means.append(y_r[mask].mean())

    ax.plot([0, 1], [0, 1], 'k--', alpha=0.5)
    ax.scatter(pred_means, obs_means, s=50, alpha=0.7)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Observed')
    ax.set_title(f'{label}: AUC={results["roc_auc"]:.3f}')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3)

fig.suptitle('Prior Model Calibration', fontsize=14)
fig.tight_layout()
fig.savefig(MODEL_DIR / 'calibration.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print(f'\nSaved calibration plot to {MODEL_DIR / "calibration.png"}')


Train:
  Accuracy:    0.6726
  Precision:   0.7424
  Recall:      0.6697
  F1:          0.7042
  ROC AUC:     0.7396
  Log Loss:    0.60138
  Brier Score: 0.20724
  Calibration:
    [0.0-0.1): pred=0.100 obs=0.000 n=1
    [0.1-0.2): pred=0.171 obs=0.167 n=639
    [0.2-0.3): pred=0.252 obs=0.287 n=2099
    [0.3-0.4): pred=0.352 obs=0.414 n=2686
    [0.4-0.5): pred=0.450 obs=0.533 n=2891
    [0.5-0.6): pred=0.549 obs=0.620 n=2983
    [0.6-0.7): pred=0.649 obs=0.733 n=2839
    [0.7-0.8): pred=0.748 obs=0.831 n=2384
    [0.8-0.9): pred=0.833 obs=0.927 n=974
    [0.9-1.0): pred=0.905 obs=0.875 n=8

Validation:
  Accuracy:    0.6432
  Precision:   0.7059
  Recall:      0.6021
  F1:          0.6499
  ROC AUC:     0.7216
  Log Loss:    0.61682
  Brier Score: 0.21459
  Calibration:
    [0.1-0.2): pred=0.171 obs=0.075 n=53
    [0.2-0.3): pred=0.248 obs=0.344 n=151
    [0.3-0.4): pred=0.356 obs=0.414 n=162
    [0.4-0.5): pred=0.450 obs=0.557 n=192
    [0.5-0.6): pred=0.547 obs=0.571 n=154
    [0

In [ ]:
# ─── 4b. Ablation: Full Model vs No-Player-Stats Model ────────────────────
#
# Train a reduced model WITHOUT player rolling stats to measure their
# incremental signal. Same hyperparameters, same splits.

PLAYER_COLS = [c for c in FEATURE_COLS if '_player_' in c]
REDUCED_COLS = [c for c in FEATURE_COLS if '_player_' not in c]

print(f"Full model features:    {len(FEATURE_COLS)}")
print(f"Player stat features:   {len(PLAYER_COLS)}")
print(f"Reduced model features: {len(REDUCED_COLS)}")
print(f"\nDropped columns ({len(PLAYER_COLS)}):")
for c in PLAYER_COLS:
    print(f"  {c}")

# Train reduced model with identical hyperparameters
X_train_r = df_train[REDUCED_COLS]
X_val_r   = df_val[REDUCED_COLS]
X_test_r  = df_test[REDUCED_COLS]

pos_rate_r = float(y_train.mean())
scale_pos_weight_r = (1.0 - pos_rate_r) / pos_rate_r

model_reduced = XGBClassifier(
    n_estimators=N_ESTIMATORS,
    learning_rate=LEARNING_RATE,
    max_depth=MAX_DEPTH,
    subsample=SUBSAMPLE,
    colsample_bytree=COLSAMPLE_BYTREE,
    reg_lambda=REG_LAMBDA,
    reg_alpha=REG_ALPHA,
    min_child_weight=MIN_CHILD_WEIGHT,
    gamma=GAMMA,
    scale_pos_weight=scale_pos_weight_r,
    random_state=RANDOM_SEED,
    n_jobs=-1,
    eval_metric='logloss',
    early_stopping_rounds=PATIENCE,
)

print(f"\nTraining REDUCED model (no player stats)...")
model_reduced.fit(
    X_train_r, y_train,
    eval_set=[(X_train_r, y_train), (X_val_r, y_val)],
    verbose=10
)

# Evaluate both models side by side
from sklearn.calibration import calibration_curve

def get_metrics(model_obj, X_tr, X_v, X_te, y_tr, y_v, y_te):
    results = {}
    for name, X, y in [('train', X_tr, y_tr), ('val', X_v, y_v), ('test', X_te, y_te)]:
        p = model_obj.predict_proba(X)[:, 1]
        results[name] = {
            'accuracy': accuracy_score(y, (p >= 0.5).astype(int)),
            'log_loss': log_loss(y, p),
            'brier':    brier_score_loss(y, p),
            'auc':      roc_auc_score(y, p),
        }
    return results

full_m    = get_metrics(model,         X_train, X_val, X_test, y_train, y_val, y_test)
reduced_m = get_metrics(model_reduced, X_train_r, X_val_r, X_test_r, y_train, y_val, y_test)

print(f"\n{'='*70}")
print(f"{'METRIC':<15} {'SPLIT':<7} {'FULL':>10} {'NO PLAYER':>10} {'DELTA':>10}")
print(f"{'='*70}")
for metric in ['accuracy', 'log_loss', 'brier', 'auc']:
    for split in ['train', 'val', 'test']:
        f_val = full_m[split][metric]
        r_val = reduced_m[split][metric]
        delta = f_val - r_val
        sign = '+' if delta > 0 else ''
        print(f"{metric:<15} {split:<7} {f_val:>10.5f} {r_val:>10.5f} {sign}{delta:>9.5f}")
    print()

print(f"Full model best round:    {model.best_iteration}")
print(f"Reduced model best round: {model_reduced.best_iteration}")

# Lower is better for log_loss and brier; higher is better for accuracy and auc
val_ll_diff = full_m['val']['log_loss'] - reduced_m['val']['log_loss']
if val_ll_diff < 0:
    print(f"\nVerdict: Player stats HELP -- val log loss improved by {-val_ll_diff:.5f}")
else:
    print(f"\nVerdict: Player stats HURT -- val log loss worsened by {val_ll_diff:.5f}")

Full model features:    159
Player stat features:   72
Reduced model features: 87

Dropped columns (72):
  away_10g_player_FGA
  home_10g_player_FGA
  away_10g_player_FG3M
  home_10g_player_FG3M
  away_10g_player_FG3A
  home_10g_player_FG3A
  away_10g_player_FTM
  home_10g_player_FTM
  away_10g_player_FTA
  home_10g_player_FTA
  away_10g_player_OREB
  home_10g_player_OREB
  away_10g_player_DREB
  home_10g_player_DREB
  away_10g_player_REB
  home_10g_player_REB
  away_10g_player_AST
  home_10g_player_AST
  away_10g_player_STL
  home_10g_player_STL
  away_10g_player_BLK
  home_10g_player_BLK
  away_10g_player_TO
  home_10g_player_TO
  away_10g_player_PF
  home_10g_player_PF
  away_10g_player_PTS
  home_10g_player_PTS
  away_10g_player_PLUS_MINUS
  home_10g_player_PLUS_MINUS
  away_10g_player_FG_PCT
  home_10g_player_FG_PCT
  away_10g_player_FG3_PCT
  home_10g_player_FG3_PCT
  away_10g_player_FT_PCT
  home_10g_player_FT_PCT
  away_20g_player_FGA
  home_20g_player_FGA
  away_20g_player_FG3

In [ ]:
# ─── 5. Feature Importance ─────────────────────────────────────────────────

importances = model.feature_importances_
fi = pd.DataFrame({
    'feature': FEATURE_COLS,
    'importance': importances
}).sort_values('importance', ascending=False)

print('Top 25 Features:')
print(fi.head(25).to_string(index=False))

# Plot top 25
fig, ax = plt.subplots(figsize=(10, 8))
top = fi.head(25).iloc[::-1]
ax.barh(top['feature'], top['importance'], color='steelblue')
ax.set_xlabel('Feature Importance')
ax.set_title('Top 25 Features - Prior Model (XGBoost)')
ax.grid(True, alpha=0.3, axis='x')
fig.tight_layout()
fig.savefig(MODEL_DIR / 'feature_importance.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print(f'\nSaved feature importance plot to {MODEL_DIR / "feature_importance.png"}')

Top 25 Features:
                   feature  importance
              home_win_pct    0.046618
              away_win_pct    0.035564
away_20g_player_PLUS_MINUS    0.029640
home_20g_player_PLUS_MINUS    0.029384
home_10g_player_PLUS_MINUS    0.026132
away_10g_player_PLUS_MINUS    0.024385
       away_20g_player_PTS    0.013658
                 home_wins    0.012446
               home_losses    0.009750
                  away_eFG    0.009739
       away_10g_player_AST    0.008767
    away_10g_player_FG_PCT    0.008749
              away_r10_eFG    0.008720
              home_r10_PTS    0.008709
   home_20g_player_FG3_PCT    0.008707
               away_losses    0.008581
       home_20g_player_PTS    0.008389
       away_10g_player_PTS    0.008370
       home_10g_player_PTS    0.008332
               home_r10_TO    0.008034
       home_10g_player_BLK    0.007922
                  home_eFG    0.007865
      away_10g_player_FG3M    0.007804
             away_r10_OREB    0.007772
      aw

In [ ]:
# ─── 6. Save Model ──────────────────────────────────

# Save model
model_path = MODEL_DIR / 'xgboost_prior.pkl'
with open(model_path, 'wb') as f:
    pickle.dump(model, f)

# Save config
config = {
    'model_type': 'XGBClassifier',
    'best_iteration': int(model.best_iteration),
    'best_val_logloss': float(model.best_score),
    'hyperparameters': {
        'n_estimators': N_ESTIMATORS,
        'learning_rate': LEARNING_RATE,
        'max_depth': MAX_DEPTH,
        'subsample': SUBSAMPLE,
        'colsample_bytree': COLSAMPLE_BYTREE,
        'reg_lambda': REG_LAMBDA,
        'min_child_weight': MIN_CHILD_WEIGHT,
        'early_stopping_rounds': PATIENCE,
    },
    'temporal_split': {
        'train_seasons': [int(s) for s in train_seasons],
        'val_season': int(val_season),
        'test_season': int(max_season),
    },
    'results': {
        'train': {k: float(v) for k, v in train_results.items() if k not in ('predictions', 'labels')},
        'val':   {k: float(v) for k, v in val_results.items()   if k not in ('predictions', 'labels')},
        'test':  {k: float(v) for k, v in test_results.items()  if k not in ('predictions', 'labels')},
    },
    'feature_cols': FEATURE_COLS,
}
with open(MODEL_DIR / 'config.json', 'w') as f:
    json.dump(config, f, indent=2)

# Save feature importances
fi.to_csv(MODEL_DIR / 'feature_importances.csv', index=False)

: 